# Somewhat Square Sudoku

[Jane Street puzzle, January 2025](https://www.janestreet.com/puzzles/somewhat-square-sudoku-index/)

> Fill the empty cells in the grid above with digits such that each row, column, and outlined 3-by-3 box contains the same set of nine unique digits, and such that the nine 9-digit numbers formed by the rows of the grid has the highest-possible GCD over any such grid.

Eight cells are pre-filled (below), so the grid uses nine of the ten digits 0–9. The answer is the middle row, read as a 9-digit number.

**Answer: 283950617** (the rows share a GCD of 12,345,679)

In [1]:
import math
from functools import reduce
from itertools import permutations

# Pre-filled cells: (row, column) -> digit, counting from 0
GIVENS = {(0, 7): 2, (1, 8): 5, (2, 1): 2, (3, 2): 0, (5, 3): 2, (6, 4): 0, (7, 5): 2, (8, 6): 5}


def show(grid):
    for row in grid:
        print(" ".join(row))


show([[str(GIVENS.get((r, c), ".")) for c in range(9)] for r in range(9)])

. . . . . . . 2 .
. . . . . . . . 5
. 2 . . . . . . .
. . 0 . . . . . .
. . . . . . . . .
. . . 2 . . . . .
. . . . 0 . . . .
. . . . . 2 . . .
. . . . . . 5 . .


## Step 1: rows that are rotations of one cycle

Write nine distinct digits in a cycle and read it from each of its nine starting points. That gives nine 9-digit numbers (leading zeros allowed), and they fit a Sudoku naturally: each column gets every digit once, and shifting the rows of a band by three places keeps the boxes valid.

Rotating also keeps common factors. Moving the first digit d of n to the end gives 10n − d × 999,999,999, so anything dividing both n and 999,999,999 divides the rotation too.

The search tries every cycle that uses 0, 2 and 5 (the pre-filled digits) and keeps the ones whose rotations have the highest GCD. Every such cycle contains 0, so each one is written starting from 0.

In [2]:
def rotations(cycle):
    return [cycle[i:] + cycle[:i] for i in range(9)]


best_gcd, best_cycles = 0, []
for missing in "1346789":
    others = [d for d in "123456789" if d != missing]
    for perm in permutations(others):
        cycle = "0" + "".join(perm)
        g = reduce(math.gcd, (int(r) for r in rotations(cycle)))
        if g > best_gcd:
            best_gcd, best_cycles = g, [cycle]
        elif g == best_gcd:
            best_cycles.append(cycle)

print(f"Highest GCD: {best_gcd:,}")
print(f"Cycles that reach it: {best_cycles}")

Highest GCD: 12,345,679
Cycles that reach it: ['098765432', '061728395', '024691358', '012345679']


## Step 2: fitting the pre-filled cells

Each pre-filled digit fixes which rotation its row must be. `fill` builds the grid a row at a time from the allowed row strings, keeping only rows that match the pre-filled cells and never repeating a digit in a column or box.

In [3]:
def fill(options):
    """Every grid built from the row strings in options that matches GIVENS and obeys the Sudoku rules."""
    per_row = [[s for s in options if all(s[c] == str(v) for (r, c), v in GIVENS.items() if r == i)]
               for i in range(9)]
    grids = []

    def place(i, grid):
        if i == 9:
            grids.append(list(grid))
            return
        for s in per_row[i]:
            column_clash = any(s[c] == grid[j][c] for j in range(i) for c in range(9))
            box_clash = any(s[c] == grid[j][cc]
                            for j in range(i - i % 3, i)
                            for c in range(9)
                            for cc in range(c - c % 3, c - c % 3 + 3))
            if not column_clash and not box_clash:
                grid.append(s)
                place(i + 1, grid)
                grid.pop()

    place(0, [])
    return grids


solutions = [grid for cycle in best_cycles for grid in fill(rotations(cycle))]
print(f"Grids that fit: {len(solutions)}\n")

grid = solutions[0]
show(grid)
print(f"\nGCD of the rows: {reduce(math.gcd, map(int, grid)):,}")
print(f"Middle row: {grid[4]}")

Grids that fit: 1

3 9 5 0 6 1 7 2 8
0 6 1 7 2 8 3 9 5
7 2 8 3 9 5 0 6 1
9 5 0 6 1 7 2 8 3
2 8 3 9 5 0 6 1 7
6 1 7 2 8 3 9 5 0
8 3 9 5 0 6 1 7 2
5 0 6 1 7 2 8 3 9
1 7 2 8 3 9 5 0 6

GCD of the rows: 12,345,679
Middle row: 283950617


## Check: no grid of any shape does better

Steps 1 and 2 only looked at grids built from rotations. To rule out anything better: every column holds the same nine digits, whose sum is S = 45 − (the missing digit). So the nine rows add up to S × 111,111,111, and any number dividing every row divides that sum.

The code takes every divisor of that sum above 12,345,679, for each possible missing digit, and searches for a grid whose rows are all multiples of it. At 12,345,679 itself it lists every grid, of any shape, whose rows are all multiples of it.

In [4]:
def divisors(n):
    small = [d for d in range(1, math.isqrt(n) + 1) if n % d == 0]
    return set(small) | {n // d for d in small}


def grids_with_rows_divisible_by(g, missing):
    digits = set("0123456789") - {missing}
    rows = [f"{m:09d}" for m in range(g, 10**9, g)]
    return fill([s for s in rows if set(s) == digits])


larger = sorted({(g, missing) for missing in "1346789"
                 for g in divisors((45 - int(missing)) * 111_111_111) if g > best_gcd}, reverse=True)
found = [(g, missing) for g, missing in larger if grids_with_rows_divisible_by(g, missing)]
print(f"Divisors above {best_gcd:,} tried: {len(larger)}. Grids found: {len(found)}")

at_best = [grid for missing in "1346789" for grid in grids_with_rows_divisible_by(best_gcd, missing)]
print(f"Grids of any shape with every row a multiple of {best_gcd:,}: {len(at_best)}")
print(f"Same grid as Step 2: {at_best == [grid]}")

Divisors above 12,345,679 tried: 102. Grids found: 0
Grids of any shape with every row a multiple of 12,345,679: 1
Same grid as Step 2: True
